# Phase 0 — Nền tảng: môi trường, CONFIG và walkthrough tutoring

**Môn:** IT701100 Mô hình đồ thị có xác suất — HK1 2026–2027 (GV: Nguyen Chi Thien, Ph.D.)
**Sinh viên:** Lê Anh Tuấn
**Đề tài:** Causal Bayesian Networks — so sánh B0 / H1 / H2 trên UCI Bank Marketing (ID 222)

Notebook này tương ứng Phase 0 trong `research-plan.md`. Nó **không phải** deliverable chấm điểm (slide 18): mục đích là
(1) dựng và ghi nhận môi trường, (2) tạo một `CONFIG` dùng chung cho mọi phase (slide 76),
(3) chạy lại toàn bộ ví dụ tutoring ở slide 3–24 để kiểm chứng pipeline *discover → identify → estimate → refute*
cho ra đúng các giá trị tham chiếu $P(Y{=}1\mid do(T{=}0)) = 0.30$, $P(Y{=}1\mid do(T{=}1)) = 0.60$, ATE $= 0.30$.

## Mục lục
1. Setup & phiên bản thư viện
2. CONFIG dùng chung
3. Tutoring: biến, DAG, CPD và giá trị tham chiếu chính xác
4. Sinh dữ liệu minh hoạ (slide 19)
5. pgmpy: fit CBN và truy vấn do(T) (slide 20)
6. Quan sát vs can thiệp (slide 11–12, 16)
7. DoWhy: identify → estimate (slide 21)
8. DoWhy GCM: mô phỏng can thiệp (slide 22)
9. DoWhy: refutation (slide 23)
10. Kiểm chứng nhiều seed (slide 24)
11. Chuẩn hoá theo cohort TEST — mẫu tính cho B0/H1/H2 (slide 17, 61)
12. Kết luận Phase 0 và việc tiếp theo


## 1. Setup & phiên bản thư viện

Chạy ô dưới nếu thiếu thư viện (Colab hoặc máy cá nhân). File `requirements.txt` nằm ở thư mục gốc dự án.

In [1]:
# %pip install -q -r ../requirements.txt
# Trên Colab: %pip install -q "pgmpy>=1.0" "dowhy>=0.12" ucimlrepo torch scikit-learn

In [2]:
import sys, platform, time, json, os, warnings, logging
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import pgmpy, dowhy, torch, sklearn

warnings.filterwarnings("ignore")
logging.getLogger("dowhy").setLevel(logging.ERROR)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ART = ROOT / "artifacts"; ART.mkdir(exist_ok=True)

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "cpu_count": os.cpu_count(),
    "cuda": torch.cuda.is_available(),
    "numpy": np.__version__, "pandas": pd.__version__, "networkx": nx.__version__,
    "pgmpy": pgmpy.__version__, "dowhy": dowhy.__version__,
    "torch": torch.__version__, "sklearn": sklearn.__version__,
}
pd.Series(ENV).to_frame("value")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
python,3.13.18
platform,Linux-6.18.44-fc-v185-x86_64-with-glibc2.39
machine,x86_64
cpu_count,2
cuda,False
numpy,2.5.3
pandas,3.0.5
networkx,3.7
pgmpy,1.1.2
dowhy,0.14


## 2. CONFIG dùng chung

Slide 76 yêu cầu *một* đối tượng cấu hình cho seed, đường dẫn, node order, state names, prior, thiết lập neural, mask và ngưỡng đồ thị.
Các khoá dành cho phase sau được khai báo sẵn (giá trị tạm) để mọi notebook sau import cùng một schema; sẽ được khoá lại ở Phase 2 trước khi đánh giá TEST.

In [3]:
CONFIG = {
    "project": {"course": "IT701100", "term": "HK1 2026-2027", "student": "Le Anh Tuan"},
    "seeds": [0, 1, 2, 3, 4],                 # >= 5 paired runs (slide 28, 68)
    "split": {"train": 0.70, "val": 0.15, "test": 0.15, "stratify_on": "y"},
    "data": {
        "uci_id": 222, "name": "Bank Marketing",
        "file": "bank-additional-full.csv",
        "raw_dir": "data/raw", "processed_dir": "data/processed",
        "treatment": "contact", "treatment_coding": {"telephone": 0, "cellular": 1},
        "outcome": "y", "outcome_coding": {"no": 0, "yes": 1},
        "drop_post_treatment": ["duration"],
    },
    "cpd": {"estimator": "BayesianEstimator", "prior_type": "BDeu", "equivalent_sample_size": 5},
    "b0": {"algorithm": "HillClimbSearch", "scoring": "bic-d", "max_indegree": 3},
    "h1": {"replace_node": "y", "hidden": [16], "dropout": 0.2, "lr": 1e-3,
            "batch_size": 256, "max_epochs": 200, "patience": 10, "loss": "bce"},
    "h2": {"method": "notears-nonlinear", "lambda1": 0.01, "lambda2": 0.01,
            "edge_threshold": 0.3, "cycle_repair": "drop-weakest-edge"},
    "refute": {"placebo": {"placebo_type": "permute", "num_simulations": 100},
               "subset": {"subset_fraction": 0.8, "num_simulations": 100}},
    "uncertainty": {"method": "bootstrap-train-fixed-test", "n_boot": 200, "level": 0.95},
    # Điền ở Phase 1–2 sau khi audit dữ liệu:
    "node_order": None, "state_names": None, "forbidden_edges": None, "baseline_covariates": None,
}

def set_seed(seed: int):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    return np.random.default_rng(seed)

(ART / "config_phase0.json").write_text(json.dumps({"CONFIG": CONFIG, "ENV": ENV}, indent=2, ensure_ascii=False))
print("saved ->", ART / "config_phase0.json")

saved -> /tmp/claude-0/-home-claude/40c863d3-96e9-559f-85ac-0ee526c2017d/scratchpad/proj/artifacts/config_phase0.json


## 3. Tutoring: biến, DAG, CPD và giá trị tham chiếu chính xác (slide 4–9, 16–17)

| Biến | Ý nghĩa | Trạng thái | Thời điểm đo |
|---|---|---|---|
| $Z$ | chuẩn bị trước (prior preparation) | 0 = thấp, 1 = cao | trước $T$ |
| $T$ | tham gia tutoring | 0 = không, 1 = có | can thiệp |
| $Y$ | kết quả thi | 0 = rớt, 1 = đậu | sau $T$ |

DAG: $Z \to T,\; Z \to Y,\; T \to Y$. Phân rã: $P(Z,T,Y) = P(Z)\,P(T\mid Z)\,P(Y\mid T,Z)$.
$Z$ là confounder (common cause). Backdoor path $T \leftarrow Z \rightarrow Y$ bị chặn khi điều chỉnh theo $\{Z\}$.

Can thiệp $do(T{=}t)$ xoá cạnh $Z \to T$ (truncated factorization): $P(z,y\mid do(T{=}t)) = P(z)\,P(y\mid t,z)$, nên
$$P(Y{=}1\mid do(T{=}t)) = \sum_z P(Y{=}1\mid t,z)\,P(z), \qquad
P(Y{=}1\mid T{=}t) = \sum_z P(Y{=}1\mid t,z)\,P(z\mid t).$$

In [4]:
# CPD tham chiếu (slide 9)
P_Z1 = 0.50
P_T1_given_Z = {0: 0.20, 1: 0.80}
P_Y1_given_TZ = {(0, 0): 0.10, (0, 1): 0.50, (1, 0): 0.40, (1, 1): 0.80}
EDGES = [("Z", "T"), ("Z", "Y"), ("T", "Y")]

def exact_reference():
    pz = {0: 1 - P_Z1, 1: P_Z1}
    rows = {}
    for t in [0, 1]:
        # P(z | t) theo Bayes
        joint_tz = {z: P_T1_given_Z[z] * pz[z] if t == 1 else (1 - P_T1_given_Z[z]) * pz[z] for z in [0, 1]}
        pt = sum(joint_tz.values())
        pz_given_t = {z: joint_tz[z] / pt for z in [0, 1]}
        obs = sum(P_Y1_given_TZ[(t, z)] * pz_given_t[z] for z in [0, 1])
        do = sum(P_Y1_given_TZ[(t, z)] * pz[z] for z in [0, 1])
        rows[t] = {"P(Z=1|T=t)": pz_given_t[1], "P(Y=1|T=t)": obs, "P(Y=1|do(T=t))": do}
    tbl = pd.DataFrame(rows).T
    tbl.loc["diff (1-0)"] = tbl.loc[1] - tbl.loc[0]
    return tbl

REF = exact_reference()
REF.round(4)   # kỳ vọng: 0.20/0.80/0.60 ; 0.18/0.72/0.54 ; 0.30/0.60/0.30 (slide 16)

,P(Z=1|T=t),P(Y=1|T=t),P(Y=1|do(T=t))
0,0.2,0.18,0.3
1,0.8,0.72,0.6
diff (1-0),0.6,0.54,0.3


In [5]:
# Cohort TEST minh hoạ (slide 17): 100 sinh viên, 40 Z=0 và 60 Z=1 -> risk(0)=0.34, risk(1)=0.64, ATE=0.30
w = {0: 0.4, 1: 0.6}
risk = {t: sum(P_Y1_given_TZ[(t, z)] * w[z] for z in [0, 1]) for t in [0, 1]}
print(f"risk(0)={risk[0]:.2f}  risk(1)={risk[1]:.2f}  ATE={risk[1]-risk[0]:.2f}")
assert abs(risk[1] - risk[0] - 0.30) < 1e-12

risk(0)=0.34  risk(1)=0.64  ATE=0.30


## 4. Sinh dữ liệu minh hoạ (slide 19)

Xác suất sinh trùng với CPD tham chiếu: $P(T{=}1\mid z)=0.2+0.6z$, $P(Y{=}1\mid t,z)=0.1+0.3t+0.4z$.

In [6]:
def generate_tutoring(n=20000, seed=42):
    rng = np.random.default_rng(seed)
    z = rng.binomial(1, 0.5, n)
    t = rng.binomial(1, 0.2 + 0.6 * z)
    y = rng.binomial(1, 0.1 + 0.3 * t + 0.4 * z)
    return pd.DataFrame({"Z": z, "T": t, "Y": y})

df = generate_tutoring(n=20000, seed=42)
print(df.groupby(["Z", "T"]).size())
print("\nTỷ lệ đậu theo (Z,T):"); print(df.groupby(["Z", "T"])["Y"].mean().round(3))

Z  T
0  0    7951
   1    1987
1  0    2028
   1    8034
dtype: int64

Tỷ lệ đậu theo (Z,T):
Z  T
0  0    0.102
   1    0.384
1  0    0.509
   1    0.806
Name: Y, dtype: float64


## 5. pgmpy: fit CBN và truy vấn do(T) (slide 20)

> **Khác biệt API:** slide 20 gọi `bn.fit(df, estimator=BayesianEstimator, prior_type="BDeu", ...)`.
> Từ pgmpy 1.1, `fit()` không nhận `prior_type` nữa; phải tạo `BayesianEstimator(...).get_parameters(prior_type=..., equivalent_sample_size=...)` rồi `add_cpds`. Hàm `fit_discrete_bn` dưới đây đóng gói lại để dùng chung cho B0/H2 ở các phase sau.

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator, MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

def fit_discrete_bn(edges, data, state_names, cfg=CONFIG["cpd"]):
    bn = DiscreteBayesianNetwork(edges)
    if cfg["estimator"] == "BayesianEstimator":
        est = BayesianEstimator(bn, data, state_names=state_names)
        cpds = est.get_parameters(prior_type=cfg["prior_type"], equivalent_sample_size=cfg["equivalent_sample_size"])
    else:  # A3 ablation: maximum likelihood
        cpds = MaximumLikelihoodEstimator(bn, data, state_names=state_names).get_parameters()
    bn.add_cpds(*cpds)
    assert bn.check_model()
    return bn

STATES = {v: [0, 1] for v in ["Z", "T", "Y"]}
t0 = time.time()
bn = fit_discrete_bn(EDGES, df, STATES)
fit_time = time.time() - t0
for cpd in bn.get_cpds():
    print(cpd, "\n")
print(f"fit time: {fit_time:.3f}s")

+------+----------+
| Z(0) | 0.496901 |
+------+----------+
| Z(1) | 0.503099 |
+------+----------+ 

+------+--------------------+--------------------+
| Z    | Z(0)               | Z(1)               |
+------+--------------------+--------------------+
| T(0) | 0.7999849102157839 | 0.2016245218341696 |
+------+--------------------+--------------------+
| T(1) | 0.2000150897842161 | 0.7983754781658304 |
+------+--------------------+--------------------+ 

+------+--------------------+-----+---------------------+
| T    | T(0)               | ... | T(1)                |
+------+--------------------+-----+---------------------+
| Z    | Z(0)               | ... | Z(1)                |
+------+--------------------+-----+---------------------+
| Y(0) | 0.8983149423119243 | ... | 0.19447123611586448 |
+------+--------------------+-----+---------------------+
| Y(1) | 0.1016850576880757 | ... | 0.8055287638841355  |
+------+--------------------+-----+---------------------+ 

fit time: 0.011

In [8]:
q = VariableElimination(bn)
pz = q.query(["Z"], show_progress=False).values

def p_do(t):
    # Backdoor adjustment theo {Z}: sum_z P(Y=1|t,z) P(z)
    return sum(q.query(["Y"], evidence={"T": t, "Z": z}, show_progress=False).values[1] * pz[z] for z in [0, 1])

def p_obs(t):
    return q.query(["Y"], evidence={"T": t}, show_progress=False).values[1]

bn_do = {t: p_do(t) for t in [0, 1]}
bn_obs = {t: p_obs(t) for t in [0, 1]}
print(f"P(Y=1|do(T=0))={bn_do[0]:.4f}  P(Y=1|do(T=1))={bn_do[1]:.4f}  ATE={bn_do[1]-bn_do[0]:.4f}   (exact 0.30 / 0.60 / 0.30)")

P(Y=1|do(T=0))=0.3065  P(Y=1|do(T=1))=0.5961  ATE=0.2896   (exact 0.30 / 0.60 / 0.30)


## 6. Quan sát vs can thiệp (slide 11–12, 16)

Chênh lệch quan sát $\approx 0.54$ trộn lẫn *selection* (nhóm tutoring có nhiều $Z{=}1$ hơn) với hiệu ứng tutoring; chênh lệch can thiệp $\approx 0.30$ mới là ATE.

In [9]:
cmp = pd.DataFrame({
    "exact": [REF.loc[0, "P(Y=1|T=t)"], REF.loc[1, "P(Y=1|T=t)"], REF.loc["diff (1-0)", "P(Y=1|T=t)"],
              REF.loc[0, "P(Y=1|do(T=t))"], REF.loc[1, "P(Y=1|do(T=t))"], REF.loc["diff (1-0)", "P(Y=1|do(T=t))"]],
    "pgmpy": [bn_obs[0], bn_obs[1], bn_obs[1] - bn_obs[0], bn_do[0], bn_do[1], bn_do[1] - bn_do[0]],
}, index=["P(Y=1|T=0)", "P(Y=1|T=1)", "obs diff", "P(Y=1|do(T=0))", "P(Y=1|do(T=1))", "ATE"])
cmp["abs err"] = (cmp["pgmpy"] - cmp["exact"]).abs()
cmp.round(4)

,exact,pgmpy,abs err
P(Y=1|T=0),0.18,0.1845,0.0045
P(Y=1|T=1),0.72,0.7219,0.0019
obs diff,0.54,0.5375,0.0025
P(Y=1|do(T=0)),0.30,0.3065,0.0065
P(Y=1|do(T=1)),0.60,0.5961,0.0039
ATE,0.30,0.2896,0.0104


## 7. DoWhy: identify → estimate (slide 21)

DoWhy phải tìm ra backdoor adjustment theo $Z$. Ước lượng tuyến tính phù hợp vì outcome probability là cộng tính trong mô hình sinh.

In [10]:
from dowhy import CausalModel

GRAPH_DOT = "digraph { Z -> T; Z -> Y; T -> Y; }"
model = CausalModel(data=df, treatment="T", outcome="Y", graph=GRAPH_DOT)
estimand = model.identify_effect(proceed_when_unidentifiable=False)
print(estimand)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
 d          
────(E[Y|Z])
d[T]        
Estimand assumption 1, Unconfoundedness: If U→{T} and U→Y then P(Y|T,Z,U) = P(Y|T,Z)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
 d          
────(E[Y|Z])
d[T]        
Estimand assumption 1, Unconfoundedness: If U→{T} and U→Y then P(Y|T,Z,U) = P(Y|T,Z)



In [11]:
t0 = time.time()
estimate = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                 control_value=0, treatment_value=1,
                                 target_units="ate", confidence_intervals=True)
dowhy_time = time.time() - t0
ci = np.asarray(estimate.get_confidence_intervals()).ravel()
print(f"DoWhy ATE = {estimate.value:.4f}   95% CI = [{ci[0]:.4f}, {ci[1]:.4f}]   ({dowhy_time:.2f}s)")

DoWhy ATE = 0.2896   95% CI = [0.2763, 0.3029]   (0.02s)


## 8. DoWhy GCM: mô phỏng can thiệp (slide 22)

Hàm hằng đặt $T=t$ cho mọi mẫu; kết quả chứa sai số fit cơ chế + sai số Monte Carlo (tuỳ chọn trong đồ án, slide 72).

In [12]:
from dowhy import gcm

cat = df.astype(str)
cm = gcm.ProbabilisticCausalModel(nx.DiGraph(EDGES))
gcm.auto.assign_causal_mechanisms(cm, cat)
gcm.fit(cm, cat)

set_seed(42)
gcm_risk = []
for t in [0, 1]:
    s = gcm.interventional_samples(cm, {"T": lambda x, t=t: str(t)}, num_samples_to_draw=20000)
    assert (s["T"] == str(t)).all()
    gcm_risk.append((s["Y"] == "1").mean())
print(f"GCM risk(0)={gcm_risk[0]:.4f}  risk(1)={gcm_risk[1]:.4f}  ATE={gcm_risk[1]-gcm_risk[0]:.4f}")

Fitting causal models:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting causal mechanism of node Z:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting causal mechanism of node T:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting causal mechanism of node Y:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting causal mechanism of node Y: 100%|██████████| 3/3 [00:00<00:00, 46.62it/s]

GCM risk(0)=0.3075  risk(1)=0.6040  ATE=0.2966


## 9. DoWhy: refutation (slide 23)

Kỳ vọng: placebo (hoán vị $T$) cho hiệu ứng $\approx 0$; subset (80 %) cho hiệu ứng gần ước lượng gốc. Lưu ý: hai kiểm tra này *refit estimator tuyến tính của DoWhy*, không kiểm định pipeline pgmpy/neural (slide 70–71).

In [13]:
R = CONFIG["refute"]
t0 = time.time()
placebo = model.refute_estimate(estimand, estimate, method_name="placebo_treatment_refuter",
                                placebo_type=R["placebo"]["placebo_type"],
                                num_simulations=R["placebo"]["num_simulations"], random_seed=42)
subset = model.refute_estimate(estimand, estimate, method_name="data_subset_refuter",
                               subset_fraction=R["subset"]["subset_fraction"],
                               num_simulations=R["subset"]["num_simulations"], random_seed=42)
refute_time = time.time() - t0
print(placebo); print(subset); print(f"refutation time: {refute_time:.1f}s")

Refute: Use a Placebo Treatment
Estimated effect:0.2896029826821583
New effect:0.0002956786380951676
p value:0.98

Refute: Use a subset of data
Estimated effect:0.2896029826821583
New effect:0.28992746113791157
p value:0.94

refutation time: 3.4s


## 10. Kiểm chứng nhiều seed (slide 24)

Lặp sinh dữ liệu + fit với 5 seed để tách *sai số lấy mẫu* khỏi *sai số cài đặt*. Cả ba ước lượng phải dao động quanh 0.30.

In [14]:
rows = []
for seed in CONFIG["seeds"]:
    d = generate_tutoring(n=20000, seed=seed)
    b = fit_discrete_bn(EDGES, d, STATES); qq = VariableElimination(b)
    pz_ = qq.query(["Z"], show_progress=False).values
    do_ = [sum(qq.query(["Y"], evidence={"T": t, "Z": z}, show_progress=False).values[1] * pz_[z] for z in [0, 1]) for t in [0, 1]]
    m = CausalModel(data=d, treatment="T", outcome="Y", graph=GRAPH_DOT)
    e = m.estimate_effect(m.identify_effect(), method_name="backdoor.linear_regression",
                          control_value=0, treatment_value=1, target_units="ate")
    rows.append({"seed": seed, "BN risk0": do_[0], "BN risk1": do_[1], "BN ATE": do_[1] - do_[0], "DoWhy ATE": e.value})
multi = pd.DataFrame(rows).set_index("seed")
summary = multi.agg(["mean", "std"]); summary.loc["exact"] = [0.30, 0.60, 0.30, 0.30]
pd.concat([multi, summary]).round(4)

,BN risk0,BN risk1,BN ATE,DoWhy ATE
0,0.3007,0.6036,0.3029,0.3029
1,0.3044,0.5970,0.2926,0.2926
2,0.3014,0.5992,0.2978,0.2978
3,0.3049,0.5962,0.2914,0.2914
4,0.3050,0.5950,0.2900,0.2900
mean,0.3033,0.5982,0.2949,0.2949
std,0.0021,0.0034,0.0053,0.0053
exact,0.3000,0.6000,0.3000,0.3000


In [15]:
assert abs(multi["BN ATE"].mean() - 0.30) < 0.02 and abs(multi["DoWhy ATE"].mean() - 0.30) < 0.02
print("OK: pipeline tái tạo đúng giá trị tham chiếu trong sai số lấy mẫu.")

OK: pipeline tái tạo đúng giá trị tham chiếu trong sai số lấy mẫu.


## 11. Chuẩn hoá theo cohort TEST — mẫu tính dùng lại cho B0/H1/H2 (slide 17, 29, 61)

Đây là phép tính cốt lõi của Table B. Với mô hình $m$ đã fit trên TRAIN và cohort TEST **cố định** có baseline $C_i$ (ở đây $C_i = Z_i$):
$$q_m(t, C_i) = P_m(Y{=}1 \mid do(T{=}t), C_i), \qquad
\text{risk}_m(t) = \frac{1}{n_\text{test}}\sum_i q_m(t, C_i), \qquad
\text{ATE}_m = \text{risk}_m(1) - \text{risk}_m(0).$$
Nhãn $Y$ của TEST **không** tham gia phép tính này; nó chỉ dùng cho Table A (AUROC, F1, Brier).

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, log_loss

d = generate_tutoring(n=1000, seed=0)                              # minh hoạ 1,000 sinh viên (slide 29)
idx_train, idx_tmp = train_test_split(d.index, test_size=0.30, random_state=0, stratify=d["Y"])
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=0, stratify=d.loc[idx_tmp, "Y"])
print(f"TRAIN={len(idx_train)}  VAL={len(idx_val)}  TEST={len(idx_test)}")

b0 = fit_discrete_bn(EDGES, d.loc[idx_train], STATES); inf = VariableElimination(b0)

def q_model(t, z):
    # Z là backdoor set hợp lệ nên q(t, z) = P(Y=1 | T=t, Z=z)
    return inf.query(["Y"], evidence={"T": t, "Z": int(z)}, show_progress=False).values[1]

test = d.loc[idx_test]
profiles = pd.DataFrame({"Z": test["Z"].values})
profiles["q(0,Z)"] = [q_model(0, z) for z in profiles["Z"]]
profiles["q(1,Z)"] = [q_model(1, z) for z in profiles["Z"]]
risk0, risk1 = profiles["q(0,Z)"].mean(), profiles["q(1,Z)"].mean()
print(profiles.drop_duplicates().sort_values("Z").round(4).to_string(index=False))
print(f"\nTEST reference: n={len(test)}, P(Z=1)={profiles['Z'].mean():.3f}")
print(f"risk(0)={risk0:.4f}  risk(1)={risk1:.4f}  ATE={risk1-risk0:.4f}")

TRAIN=700  VAL=150  TEST=150


 Z  q(0,Z)  q(1,Z)
 0  0.1026  0.3456
 1  0.4877  0.7683

TEST reference: n=150, P(Z=1)=0.613
risk(0)=0.3388  risk(1)=0.6049  ATE=0.2661


In [17]:
# Table A kiểu mẫu: dự đoán Y trên TEST từ evidence (T, Z) -> so với Y quan sát
p_hat = np.array([inf.query(["Y"], evidence={"T": int(t), "Z": int(z)}, show_progress=False).values[1]
                  for t, z in zip(test["T"], test["Z"])])
thr = 0.5  # ngưỡng F1 phải chọn trên VALIDATION ở đồ án thật (slide 74)
tableA = pd.Series({
    "AUROC": roc_auc_score(test["Y"], p_hat),
    "macro-F1": f1_score(test["Y"], (p_hat >= thr).astype(int), average="macro"),
    "Brier": brier_score_loss(test["Y"], p_hat),
    "log loss": log_loss(test["Y"], p_hat),
    "test n": len(test), "positive rate": test["Y"].mean(),
}, name="B0 (tutoring demo)")
tableA.round(4)

AUROC              0.8167
macro-F1           0.7792
Brier              0.1642
log loss           0.4951
test n           150.0000
positive rate      0.4333
Name: B0 (tutoring demo), dtype: float64

## 12. Kết luận Phase 0 và việc tiếp theo

**Đã xác nhận**
- Môi trường chạy được pgmpy ≥ 1.1, DoWhy 0.14, PyTorch; phiên bản và phần cứng lưu ở `artifacts/config_phase0.json`.
- Pipeline tutoring tái tạo đúng 0.30 / 0.60 / ATE 0.30 bằng ba đường độc lập (pgmpy adjustment, DoWhy linear backdoor, GCM simulation), placebo ≈ 0, subset ≈ ước lượng gốc.
- Đã có hàm dùng chung: `set_seed`, `fit_discrete_bn` (BDeu hoặc ML cho A3), mẫu chuẩn hoá theo cohort TEST và Table A.
- Lưu ý API: pgmpy 1.1 không nhận `prior_type` trong `fit()` — đã đóng gói trong `fit_discrete_bn`.

**Phase 1 (tiếp theo)** — `01_bank_marketing_cbn.ipynb`, mục 1–2:
- Tải `bank-additional-full.csv` (UCI ID 222) vào `data/raw/`, ghi ngày tải, checksum, license.
- Variable dictionary + bảng timing; loại `duration`; kiểm tra overlap `contact × month / poutcome / age-bin`.
- Khoá `CONFIG["node_order"]`, `state_names`, `forbidden_edges`, `baseline_covariates`.
